In [4]:
import numpy as np    
import pandas as pd
from sklearn.preprocessing import LabelEncoder


In [2]:
df=pd.read_csv('cleaned_land_merged_final_after_eda.csv')

In [5]:
df_ml_land = df.copy()
         

In [6]:
# ─────────────────────────────────────────
# STEP 1 — DROP UNWANTED COLUMNS
# ─────────────────────────────────────────
drop_cols = [
    'category', 'location_raw',
    'is_price_suspect', 'is_price_outlier',
    'is_rate_outlier', 'calculated_total_price',
    'price_segment'
]
df_ml_land.drop(columns=drop_cols, inplace=True, errors='ignore')

In [7]:
# ─────────────────────────────────────────
# STEP 2 — LOG TRANSFORM
# ─────────────────────────────────────────
df_ml_land['log_land'] = np.log1p(df_ml_land['land_size_aana'])

In [9]:
# ─────────────────────────────────────────
# STEP 3 — ENCODE CATEGORICALS
# ─────────────────────────────────────────
le = LabelEncoder()

for col in ['district', 'road_type', 'facing', 'source']:
    df_ml_land[col] = le.fit_transform(df_ml_land[col].astype(str))

In [10]:
# Target encode neighborhood
target_map  = df_ml_land.groupby('neighborhood')['price_per_aana'].apply(
    lambda x: np.log1p(x).mean()
)
global_mean = np.log1p(df_ml_land['price_per_aana']).mean()
df_ml_land['neighborhood_encoded'] = df_ml_land['neighborhood'].map(target_map).fillna(global_mean)
df_ml_land.drop(columns=['neighborhood'], inplace=True)


In [11]:
# ─────────────────────────────────────────
# STEP 4 — BOOLEAN TO INT
# ─────────────────────────────────────────
bool_cols = df_ml_land.select_dtypes(include='bool').columns.tolist()
df_ml_land[bool_cols] = df_ml_land[bool_cols].astype(int)

In [12]:
# ─────────────────────────────────────────
# STEP 5 — VERIFY
# ─────────────────────────────────────────
print("=== Object columns remaining ===")
obj = df_ml_land.select_dtypes(include='object').columns.tolist()
print(obj if obj else "✅ None")

print("\n=== Nulls ===")
nulls = df_ml_land.isnull().sum()
print(nulls[nulls > 0] if nulls[nulls > 0].any() else "✅ No nulls")

print(f"\n=== Final Shape: {df_ml_land.shape} ===")
print(f"=== Columns: {list(df_ml_land.columns)} ===")

=== Object columns remaining ===
✅ None

=== Nulls ===
✅ No nulls

=== Final Shape: (4063, 11) ===
=== Columns: ['district', 'road_type', 'land_size_aana', 'price_per_aana', 'is_large_plot', 'road_width_feet', 'is_wide_road', 'facing', 'source', 'log_land', 'neighborhood_encoded'] ===


In [13]:
# ─────────────────────────────────────────
# STEP 6 — SAVE
# ─────────────────────────────────────────
df_ml_land.to_csv('land_data_after_feature_engineering_and_encoding.csv', index=False)
print("\n✅ Feature engineered land dataset saved")


✅ Feature engineered land dataset saved
